In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [2]:
import torch
print(torch.cuda.device_count())   

1


In [3]:
from datasets import load_dataset

data = load_dataset("json", data_files="data/my_train.jsonl", split="train")
data = data.train_test_split(test_size=0.1, seed=42)
print(data)

/home/gusvahaye@GU.GU.SE/Swedish-Grammaticality-thesis/venv/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['text', 'score'],
        num_rows: 14045
    })
    test: Dataset({
        features: ['text', 'score'],
        num_rows: 1561
    })
})


## tokenization

In [4]:
from transformers import AutoTokenizer

model_name = "AI-Sweden-Models/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [5]:
enc = tokenizer("Den gamla bron renoverades.")
enc

{'input_ids': [3, 1685, 3796, 22876, 25324, 706, 20, 4], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}

In [6]:
tokenizer.convert_ids_to_tokens(enc["input_ids"])

['[CLS]', 'Den', 'Ġgamla', 'Ġbron', 'Ġrenovera', 'des', '.', '[SEP]']

In [7]:
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

data = data.map(tokenize, batched= True)

In [8]:
data = data.rename_column("score", "labels")

In [9]:
print(data)

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 14045
    })
    test: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1561
    })
})


In [10]:
data = data.remove_columns(["token_type_ids"])

In [11]:
print(data)

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 14045
    })
    test: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 1561
    })
})


## loading the model

In [12]:
from transformers import set_seed
set_seed(42)

In [13]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1, reference_compile=False)

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at AI-Sweden-Models/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
n_params = sum(p.numel() for p in model.parameters())
n_params

149605633

first trying the untrained model

In [15]:
for sentence in ["Den gamla bron renoverades.", "Den gamla bro renoverades."]:
    enc = tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        out = model(input_ids=enc["input_ids"], attention_mask = enc["attention_mask"])
        print(sentence, out.logits.item())


Den gamla bron renoverades. -0.5433064103126526
Den gamla bro renoverades. 0.650090217590332


## now setting up the trainer

In [16]:
from scipy.stats import spearmanr

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.squeeze()
    result = spearmanr(predictions, labels)
    return {"spearmanr": result.correlation}

In [17]:
# settings
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="checkpoints",
    num_train_epochs= 3,
    learning_rate= 2e-5,
    per_device_train_batch_size= 32, 
    eval_strategy= "epoch",
    logging_steps=50,
    seed=42,
    report_to="none",
    warmup_ratio=0.1
)

In [18]:
from transformers import Trainer, DataCollatorWithPadding

trainer = Trainer(
    model=model,
    args= args,
    train_dataset=data["train"],
    eval_dataset=data["test"],
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

In [19]:
trainer.train_dataset = data["train"]
trainer.train()

/home/gusvahaye@GU.GU.SE/Swedish-Grammaticality-thesis/venv/lib/python3.12/site-packages/triton/backends/nvidia/driver.c:9:10: fatal error: Python.h: No such file or directory
    9 | #include <Python.h>
      |          ^~~~~~~~~~
compilation terminated.


Epoch,Training Loss,Validation Loss,Spearmanr
1,0.109000,0.085455,0.036393
2,0.095700,0.085551,0.345157
3,0.055400,0.056907,0.512346


TrainOutput(global_step=1317, training_loss=0.15527321397126762, metrics={'train_runtime': 334.2912, 'train_samples_per_second': 126.043, 'train_steps_per_second': 3.94, 'total_flos': 2167125011434800.0, 'train_loss': 0.15527321397126762, 'epoch': 3.0})

In [20]:
# saving the model
trainer.save_model("my_model")
tokenizer.save_pretrained("my_model")

('my_model/tokenizer_config.json',
 'my_model/special_tokens_map.json',
 'my_model/tokenizer.json')

In [21]:
# test
model.eval()
for s in ["Den gamla bron renoverades.", "Den gamla bro renoverades."]:
    inputs = tokenizer(s, return_tensors="pt").to(model.device)
    with torch.no_grad():
        print(s, model(**inputs).logits.item())

Den gamla bron renoverades. 0.9368273019790649
Den gamla bro renoverades. 0.5194348692893982


---

## analysis



| Metric   | Run 1 (DaLAJ + toy) | Run 2 (+ScaLA) | Run 3 (+warmup, lr = 2e-5) |
|-----------|--------------------|----------------|----------------------------|
| Val loss  | 0.0765             | 0.0773         | 0.0568                     |
| Spearman  | 0.05               | -0.06          | 0.50                       |

so it is doing what a lazy model would do so collapsing to the mean

In [22]:
# check 1 does it give everyone the same score
out = trainer.predict(data["test"])
preds = out.predictions.squeeze()
print(preds.min(), preds.max(), preds.std())

0.23545174 1.0491596 0.15651034


In [23]:
ex = data["train"][0]
print(tokenizer.decode(ex["input_ids"]))
print(ex["labels"])
print(data["train"].column_names)

[CLS]En person kan röra sig mellan olika rollförväntningar, när han flyttar sig mellan grupper som familjen, arbetsgruppen, gänget, idrottsklubben etc i vilka han intar olika status.[SEP]
0.95
['text', 'labels', 'input_ids', 'attention_mask']


In [24]:
for log in trainer.state.log_history[:10]:
    print(log)

{'loss': 1.4749, 'grad_norm': 130.2480010986328, 'learning_rate': 7.424242424242425e-06, 'epoch': 0.11389521640091116, 'step': 50}
{'loss': 0.2152, 'grad_norm': 56.313194274902344, 'learning_rate': 1.5000000000000002e-05, 'epoch': 0.22779043280182232, 'step': 100}
{'loss': 0.1482, 'grad_norm': 90.30645751953125, 'learning_rate': 1.9713080168776375e-05, 'epoch': 0.3416856492027335, 'step': 150}
{'loss': 0.1866, 'grad_norm': 123.33102416992188, 'learning_rate': 1.8869198312236287e-05, 'epoch': 0.45558086560364464, 'step': 200}
{'loss': 0.1123, 'grad_norm': 67.19342803955078, 'learning_rate': 1.8025316455696206e-05, 'epoch': 0.5694760820045558, 'step': 250}
{'loss': 0.1712, 'grad_norm': 54.70029067993164, 'learning_rate': 1.718143459915612e-05, 'epoch': 0.683371298405467, 'step': 300}
{'loss': 0.1286, 'grad_norm': 54.54928207397461, 'learning_rate': 1.6337552742616034e-05, 'epoch': 0.7972665148063781, 'step': 350}
{'loss': 0.109, 'grad_norm': 72.25599670410156, 'learning_rate': 1.54936708